# 11.7 怎麼維持文字能力？


圖文訓練後模型更會認顏色，卻開始把純文字2+2答錯。這不是圖片入口的形狀問題，而是新任務改變了共享語言參數，使舊技能退步。本節把舊文字例混進訓練，同時把兩類成績並排看，避免只看圖文代價下降。

前置是[7.15舊技能遺忘](https://birdhackor.github.io/tiny-perceptron-vlm/7.15.html)、[7.16舊資料回放](https://birdhackor.github.io/tiny-perceptron-vlm/7.16.html)與[10.8純文字入口一致性](https://birdhackor.github.io/tiny-perceptron-vlm/10.8.html)。10.8確認訓練前包裝沒有改變計算；這裡稱為retention的能力保留，檢查訓練後的行為是否仍會做原任務。兩件事發生在不同階段，不能用前者通過代替後者。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
recipe = {"文字有效位置": 750, "圖文有效位置": 250}
recorded = {
    "訓練前": {"文字答對": 8, "圖文答對": 2},
    "訓練後": {"文字答對": 6, "圖文答對": 7},
}
print("示例訓練比例", {k: v / sum(recipe.values()) for k, v in recipe.items()})
for phase, scores in recorded.items():
    print(phase, "文字", scores["文字答對"] / 10, "圖文", scores["圖文答對"] / 10)

輸入recipe是計劃中1000個有效回答位置的分配，不是同樣數量的完整對話；文字佔0.75、圖文佔0.25。recorded是人工構造的成績示例，兩類各有10道固定測試題。輸出前後文字0.8到0.6，圖文0.2到0.7。這樣並排就看見新任務進步與舊任務退步同時存在。

這不是宣稱75%文字回放仍必定遺忘：程式沒有訓練，數字用來演示報告方法。實際要建立不進入訓練的文字測試，固定題目與評分規則，在基模和多模態權重上各跑一遍。還可以按算術、格式、一般文字等子任務拆開，看退步集中在哪裡。訓練時用目標答案衡量預測的猜錯代價，稱為loss；通常越低表示對訓練目標的預測越吻合，具體計算可看[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)。固定測試的答對率則是答對題數除以總題數，例如8/10。本段程式只整理人工成績，沒有計算loss。混合後的總loss較低，也可能只是較容易的那組佔比增加，並不保證所有技能都改善。

選擇回放比例時，要保持總有效位置預算不變，否則加文字同時延長訓練，會把比例與總資源混在一起。[接頭（projector）](https://birdhackor.github.io/tiny-perceptron-vlm/10.6.html)是圖片與文字模型之間的轉換層，把圖片特徵轉成文字模型要求的特徵個數。[語言權重凍結](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)指訓練時固定這些參數的數值、不更新它們，仍讓它們照常參與計算。若只有圖片接頭更新，純文字輸入使用的語言參數維持原值，文字路徑按理應更容易保持；仍需核對完整應用的提示或前處理是否也維持一致。開放更多語言層更新則提供更多調整空間，也增加檢查舊能力的必要。

示例中的文字下降是8/10到6/10，不一定有足夠樣本判斷每次訓練都真的退步。recorded只保存答對總數，沒有逐題回答；淨少答對兩題，可能同時有些題改對、更多題改錯，不能推出恰好新增兩道錯題。實測時把前後逐題回答並排，找出原先答對、後來答錯的題目，再確認失誤類型。若兩份模型生成有隨機性，應固定選擇方式或重複測量，避免把一次抽樣差異當遺忘。圖文7/10也有三道失敗，仍需查圖像前處理與題目類別。並排表的價值是讓這些問題有入口，而不是用一個較漂亮的新任務平均分結束討論。

練習只把recipe改成文字500、圖文500，先預測比例各0.5而總數仍1000，再執行核對。不要改recorded成績：比例計劃改變不能憑空產生測量結果。要知道新比例是否改善保留，必須訓練後重新填入真實測試成績，未量測的格子應標為未知。
